In [5]:
# Day 17 - LoRA Fine-tuning (PEFT)
!pip install -q peft accelerate bitsandbytes trl datasets

import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

print("✅ 4-bit quantized model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ 4-bit quantized model loaded successfully!


In [6]:
# 2. Create Better Dataset
data = [
    {"text": "Question: What is the capital of Ethiopia?\nAnswer: The capital of Ethiopia is Addis Ababa."},
    {"text": "Question: What is the national dish of Ethiopia?\nAnswer: The national dish is Injera, made from teff flour."},
    {"text": "Question: Where was coffee discovered?\nAnswer: Coffee was discovered in Ethiopia."},
    {"text": "Question: Tell me about Lalibela.\nAnswer: Lalibela is famous for its rock-hewn churches."},
    {"text": "Question: What is special about Ethiopian culture?\nAnswer: Ethiopia has one of the oldest continuous civilizations and rich history."},
]

dataset = Dataset.from_list(data)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [7]:
# 3. LoRA Config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [10]:


from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer


from transformers import DataCollatorForLanguageModeling
training_args = TrainingArguments(
    output_dir="lora_finetuned_model",
    num_train_epochs=3,                    # Reduced for faster training
    per_device_train_batch_size=2,
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_8bit",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=9, training_loss=1.7433435651991103, metrics={'train_runtime': 9.7406, 'train_samples_per_second': 1.54, 'train_steps_per_second': 0.924, 'total_flos': 23887069839360.0, 'train_loss': 1.7433435651991103, 'epoch': 3.0})